# TVSum and SumMe H5-Exact Embedding Rebuild

This notebook rebuilds the benchmark pipeline directly from the ECCV16 H5 files. It uses the H5 metadata as the source of truth for:

1. dataset ordering and video mapping,
2. video duration checks,
3. frame selection through `picks`,
4. embedding generation for exactly those selected frames,
5. H5 reconstruction for each dataset and model.

The Drive paths are left unchanged.

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# ============================================================
# CELL 1 - Setup and shared helpers
# ============================================================
import sys
import subprocess # Moved import to the top
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'h5py', 'pandas', 'numpy', 'pillow', 'tqdm', 'matplotlib', 'seaborn', 'open_clip_torch'], check=True)

from pathlib import Path
import math
import json
import shutil
from difflib import SequenceMatcher
from typing import Any, Dict, List, Optional, Tuple

import h5py
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from torchvision import transforms
from tqdm.auto import tqdm

try:
    import open_clip
except ImportError as exc:
    raise ImportError('open_clip_torch is required for the CLIP models') from exc

DRIVE_ROOT = Path('/content/drive/MyDrive/4th year/S2/RL/Project')

TVSUM_VIDEO_DIR = DRIVE_ROOT / 'Dataset' / 'TVSUM'
TVSUM_H5 = TVSUM_VIDEO_DIR / 'eccv16_dataset_tvsum_google_pool5.h5'
TVSUM_INFO_TSV = DRIVE_ROOT / 'Dataset' / 'TVSUM' / 'ydata-tvsum50-info.tsv'

SUMME_VIDEO_DIR = DRIVE_ROOT / 'Dataset' / 'SUMME'
SUMME_H5 = SUMME_VIDEO_DIR / 'eccv16_dataset_summe_google_pool5.h5'

OUTPUT_ROOT = DRIVE_ROOT / 'benchmark_embeddings'
FRAME_ROOT = OUTPUT_ROOT / 'frames_2fps'
EMBEDDINGS_ROOT = OUTPUT_ROOT / 'embeddings'
MANIFEST_ROOT = OUTPUT_ROOT / 'manifests'
H5_OUT_ROOT = OUTPUT_ROOT.parent / 'h5_files'

FPS = 2
BATCH_SIZE = 64
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

for path in [OUTPUT_ROOT, FRAME_ROOT, EMBEDDINGS_ROOT, MANIFEST_ROOT, H5_OUT_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}')
print(f'ffmpeg: {shutil.which("ffmpeg") or "not found"}')
print(f'Drive root: {DRIVE_ROOT}')
print(f'TVSum video dir: {TVSUM_VIDEO_DIR}')
print(f'SumMe video dir: {SUMME_VIDEO_DIR}')
print(f'Output root: {OUTPUT_ROOT}')

MODEL_SPECS = {
    'clip_vitl14': {
        'kind': 'clip',
        'arch': 'ViT-L-14',
        'pretrained': 'openai',
        'expected_dim': 768,
    },
    'clip_rn50x64': {
        'kind': 'clip',
        'arch': 'RN50x64',
        'pretrained': 'openai',
        'expected_dim': 1024,
    },
    'dinov2_vitl14': {
        'kind': 'dinov2',
        'arch': 'dinov2_vitl14',
        'pretrained': None,
        'expected_dim': 1024,
    },
}


def decode_scalar(value: Any) -> Any:
    if isinstance(value, bytes):
        return value.decode('utf-8')
    if hasattr(value, 'item'):
        value = value.item()
    return value


def h5_sorted_keys(h5_file: h5py.File) -> List[str]:
    def key_rank(name: str) -> int:
        suffix = name.split('_')[-1]
        return int(suffix) if suffix.isdigit() else 10**9
    return sorted(list(h5_file.keys()), key=key_rank)


def ffprobe_duration_seconds(video_path: Path) -> Optional[float]:
    try:
        result = subprocess.run(
            [
                'ffprobe',
                '-v', 'error',
                '-show_entries', 'format=duration',
                '-of', 'default=noprint_wrappers=1:nokey=1',
                str(video_path),
            ],
            capture_output=True,
            text=True,
            timeout=20,
        )
        if result.returncode == 0 and result.stdout.strip():
            return float(result.stdout.strip())
    except Exception:
        pass
    return None


def ffprobe_fps(video_path: Path) -> Optional[float]:
    try:
        result = subprocess.run(
            [
                'ffprobe',
                '-v', 'error',
                '-select_streams', 'v:0',
                '-show_entries', 'stream=r_frame_rate',
                '-of', 'default=noprint_wrappers=1:nokey=1',
                str(video_path),
            ],
            capture_output=True,
            text=True,
            timeout=20,
        )
        if result.returncode == 0 and result.stdout.strip():
            num, den = result.stdout.strip().split('/')
            return float(num) / float(den)
    except Exception:
        pass
    return None


def fuzzy_match(name1: str, name2: str, threshold: float = 0.8) -> bool:
    norm1 = name1.lower().replace('_', ' ').replace('-', ' ')
    norm2 = name2.lower().replace('_', ' ').replace('-', ' ')
    return SequenceMatcher(None, norm1, norm2).ratio() >= threshold


def resolve_video_path(video_dir: Path, canonical_name: str) -> Path:
    direct = video_dir / f'{canonical_name}.mp4'
    if direct.exists():
        return direct
    matches = list(video_dir.glob(f'{canonical_name}*.mp4'))
    if matches:
        return matches[0]
    for candidate in sorted(video_dir.glob('*.mp4')):
        if fuzzy_match(canonical_name, candidate.stem):
            return candidate
    return direct


def clean_output_tree(confirm: bool = False) -> None:
    """
    Safely handle existing output directories. This function will NOT delete
    anything unless `confirm=True` is passed. When `confirm=True` the
    directories are moved to a timestamped backup folder under
    `OUTPUT_ROOT.parent / 'backups'` to avoid permanent deletion.
    """
    targets = [FRAME_ROOT, EMBEDDINGS_ROOT, MANIFEST_ROOT, H5_OUT_ROOT]
    if not confirm:
        print('clean_output_tree called without confirm=True; skipping.')
        print('Existing output directories (no changes):')
        for p in targets:
            print(' -', p, '(exists)' if p.exists() else '(missing)')
        print('\nTo backup current outputs call: clean_output_tree(confirm=True)')
        return

    timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
    backup_root = OUTPUT_ROOT.parent / 'backups' / timestamp
    backup_root.mkdir(parents=True, exist_ok=True)

    for p in targets:
        if p.exists():
            dest = backup_root / p.name
            try:
                shutil.move(str(p), str(dest))
                print(f'Moved {p} -> {dest}')
            except Exception as exc:
                print(f'Could not move {p}: {exc}')

    # Recreate empty output dirs
    for path in [OUTPUT_ROOT, FRAME_ROOT, EMBEDDINGS_ROOT, MANIFEST_ROOT, H5_OUT_ROOT]:
        path.mkdir(parents=True, exist_ok=True)


def expected_frame_indices(picks: np.ndarray, native_fps: float, duration_limit_sec: float) -> np.ndarray:
    max_frame_index = int(math.floor(duration_limit_sec * native_fps + 1e-6))
    picks = np.asarray(picks, dtype=np.int32)
    picks = np.unique(picks)
    picks.sort()
    return picks[picks <= max_frame_index]


def save_json(path: Path, payload: Dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')


Device: cuda
ffmpeg: /usr/bin/ffmpeg
Drive root: /content/drive/MyDrive/4th year/S2/RL/Project
TVSum video dir: /content/drive/MyDrive/4th year/S2/RL/Project/Dataset/TVSUM
SumMe video dir: /content/drive/MyDrive/4th year/S2/RL/Project/Dataset/SUMME
Output root: /content/drive/MyDrive/4th year/S2/RL/Project/benchmark_embeddings


In [6]:
# ============================================================
# CELL 2 - Build dataset contracts from the H5 files
# ============================================================

def build_tvsum_entries() -> List[Dict]:
    tsv = pd.read_csv(TVSUM_INFO_TSV, sep='\t')
    if 'video_id' not in tsv.columns or 'length' not in tsv.columns:
        raise ValueError(f'Unexpected TVSum TSV columns: {list(tsv.columns)}')

    with h5py.File(TVSUM_H5, 'r') as h5:
        keys = h5_sorted_keys(h5)
        if len(keys) != len(tsv):
            print(f'[WARN] TVSum TSV rows ({len(tsv)}) != H5 keys ({len(keys)})')

        entries = []
        for h5_key, (_, row) in zip(keys, tsv.iterrows()):
            video_id = str(row['video_id'])
            source_path = resolve_video_path(TVSUM_VIDEO_DIR, video_id)
            group = h5[h5_key]
            n_frames = int(group['n_frames'][()])
            n_steps = int(group['n_steps'][()])
            picks = group['picks'][()].astype(np.int32)
            actual_duration_sec = ffprobe_duration_seconds(source_path)
            native_fps = ffprobe_fps(source_path)
            if actual_duration_sec is None or native_fps is None:
                raise RuntimeError(f'Could not probe TVSum video: {source_path}')

            # Convert 'MM:SS' string to seconds (float)
            length_str = str(row['length'])
            if ':' in length_str:
                minutes, seconds = map(int, length_str.split(':'))
                reference_duration_sec = float(minutes * 60 + seconds)
            else:
                reference_duration_sec = float(length_str)

            duration_limit_sec = min(actual_duration_sec, reference_duration_sec)

            entries.append({
                'dataset': 'tvsum',
                'h5_key': h5_key,
                'canonical_name': video_id,
                'source_path': source_path,
                'native_fps': native_fps,
                'actual_duration_sec': actual_duration_sec,
                'reference_duration_sec': reference_duration_sec,
                'duration_limit_sec': duration_limit_sec,
                'n_frames': n_frames,
                'n_steps': n_steps,
                'picks': picks,
            })
    return entries


def build_summe_entries() -> List[Dict]:
    with h5py.File(SUMME_H5, 'r') as h5:
        entries = []
        for h5_key in h5_sorted_keys(h5):
            group = h5[h5_key]
            canonical_name = decode_scalar(group['video_name'][()]) if 'video_name' in group else h5_key
            source_path = resolve_video_path(SUMME_VIDEO_DIR, canonical_name)
            n_frames = int(group['n_frames'][()])
            n_steps = int(group['n_steps'][()])
            picks = group['picks'][()].astype(np.int32)
            actual_duration_sec = ffprobe_duration_seconds(source_path)
            native_fps = ffprobe_fps(source_path)
            if actual_duration_sec is None or native_fps is None:
                raise RuntimeError(f'Could not probe SumMe video: {source_path}')

            reference_duration_sec = actual_duration_sec
            duration_limit_sec = min(actual_duration_sec, reference_duration_sec)

            entries.append({
                'dataset': 'summe',
                'h5_key': h5_key,
                'canonical_name': canonical_name,
                'source_path': source_path,
                'native_fps': native_fps,
                'actual_duration_sec': actual_duration_sec,
                'reference_duration_sec': reference_duration_sec,
                'duration_limit_sec': duration_limit_sec,
                'n_frames': n_frames,
                'n_steps': n_steps,
                'picks': picks,
            })
    return entries


def audit_contract(entries: List[Dict], dataset_name: str) -> pd.DataFrame:
    h5_path = TVSUM_H5 if dataset_name == 'tvsum' else SUMME_H5
    rows = []
    for entry in entries:
        with h5py.File(h5_path, 'r') as h5:
            group = h5[entry['h5_key']]
            h5_n_frames = int(group['n_frames'][()])
            h5_n_steps = int(group['n_steps'][()])

        duration_matches = True
        if dataset_name == 'tvsum':
            # Allow a one-second tolerance between probed duration and TSV reference
            try:
                duration_matches = abs(float(entry['actual_duration_sec']) - float(entry['reference_duration_sec'])) <= 1.0
            except Exception:
                duration_matches = False

        rows.append({
            'dataset': dataset_name,
            'h5_key': entry['h5_key'],
            'canonical_name': entry['canonical_name'],
            'fps': entry['native_fps'],
            'actual_duration_sec': entry['actual_duration_sec'],
            'reference_duration_sec': entry['reference_duration_sec'],
            'duration_matches': duration_matches,
            'h5_n_frames': h5_n_frames,
            'h5_n_steps': h5_n_steps,
            'picks_len': len(entry['picks']),
            'source_path': str(entry['source_path']),
        })

    df = pd.DataFrame(rows)
    print()
    print(f'{dataset_name.upper()} contract audit')
    print(f'  videos checked      : {len(df)}')
    print(f'  duration matches    : {int(df["duration_matches"].sum())}/{len(df)}')
    print(f'  fps range           : {df["fps"].min():.3f} .. {df["fps"].max():.3f}')
    print(f'  n_steps range       : {df["h5_n_steps"].min()} .. {df["h5_n_steps"].max()}')
    bad = df[~df['duration_matches']]
    if len(bad):
        print('  mismatches:')
        print(bad[['h5_key', 'canonical_name', 'actual_duration_sec', 'reference_duration_sec']].to_string(index=False))
    return df


tvsum_entries = build_tvsum_entries()
summe_entries = build_summe_entries()

tvsum_contract_df = audit_contract(tvsum_entries, 'tvsum')
summe_contract_df = audit_contract(summe_entries, 'summe')

display(tvsum_contract_df.head())
display(summe_contract_df.head())


TVSUM contract audit
  videos checked      : 50
  duration matches    : 50/50
  fps range           : 23.976 .. 30.000
  n_steps range       : 167 .. 1294

SUMME contract audit
  videos checked      : 25
  duration matches    : 25/25
  fps range           : 30.000 .. 30.000
  n_steps range       : 64 .. 649


,dataset,h5_key,canonical_name,fps,actual_duration_sec,reference_duration_sec,duration_matches,h5_n_frames,h5_n_steps,picks_len,source_path
0,tvsum,video_1,AwmHb44_ouw,29.970030,353.640,354.0,True,10597,707,707,/content/drive/MyDrive/4th year/S2/RL/Project/...
1,tvsum,video_2,98MoyGZKHXc,25.000000,187.571,187.0,True,4688,313,313,/content/drive/MyDrive/4th year/S2/RL/Project/...
2,tvsum,video_3,J0nA4VgnoCo,23.976024,584.772,584.0,True,14019,935,935,/content/drive/MyDrive/4th year/S2/RL/Project/...
3,tvsum,video_4,gzDbaEs1Rlg,25.000000,288.462,288.0,True,7210,481,481,/content/drive/MyDrive/4th year/S2/RL/Project/...
4,tvsum,video_5,XzYM3PfTM4w,29.970030,111.062,111.0,True,3327,222,222,/content/drive/MyDrive/4th year/S2/RL/Project/...


,dataset,h5_key,canonical_name,fps,actual_duration_sec,reference_duration_sec,duration_matches,h5_n_frames,h5_n_steps,picks_len,source_path
0,summe,video_1,Air_Force_One,30.0,149.800,149.800,True,4494,300,300,/content/drive/MyDrive/4th year/S2/RL/Project/...
1,summe,video_2,Base jumping,30.0,157.634,157.634,True,4729,316,316,/content/drive/MyDrive/4th year/S2/RL/Project/...
2,summe,video_3,Bearpark_climbing,30.0,111.367,111.367,True,3341,223,223,/content/drive/MyDrive/4th year/S2/RL/Project/...
3,summe,video_4,Bike Polo,30.0,102.134,102.134,True,3064,205,205,/content/drive/MyDrive/4th year/S2/RL/Project/...
4,summe,video_5,Bus_in_Rock_Tunnel,30.0,171.200,171.200,True,5131,343,343,/content/drive/MyDrive/4th year/S2/RL/Project/...


## 3. Extract exact frames and verify them before embedding

The frame directories are rebuilt from the H5 `picks` arrays. Each video is truncated to the dataset reference duration before the final frame list is checked. The extraction step fails fast if the saved frame indices do not match `picks` exactly.

In [7]:
# ============================================================
# CELL 3 - Exact frame extraction and verification
# ============================================================

def _frame_manifest_path(output_dir: Path) -> Path:
    return output_dir / 'frames_meta.json'


def _load_frame_manifest(output_dir: Path) -> Optional[Dict]:
    meta_path = _frame_manifest_path(output_dir)
    if not meta_path.exists():
        return None
    try:
        return json.loads(meta_path.read_text(encoding='utf-8'))
    except Exception:
        return None


def _save_frame_manifest(output_dir: Path, meta: Dict) -> None:
    meta_path = _frame_manifest_path(output_dir)
    meta_path.parent.mkdir(parents=True, exist_ok=True)
    meta_path.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding='utf-8')


def extract_exact_frames(video_path: Path, output_dir: Path, picks: np.ndarray, native_fps: float, duration_limit_sec: float, overwrite: bool = True) -> List[Path]:
    output_dir.mkdir(parents=True, exist_ok=True)

    expected_picks = expected_frame_indices(picks, native_fps, duration_limit_sec)
    expected_indices = expected_picks.astype(int).tolist()

    # Check existing manifest to allow resuming
    existing_manifest = _load_frame_manifest(output_dir)
    if existing_manifest and not overwrite:
        try:
            stored_fps = float(existing_manifest.get('native_fps', 0.0))
            stored_expected = existing_manifest.get('expected_indices', [])
            stored_saved = existing_manifest.get('saved_indices', [])
            if abs(stored_fps - float(native_fps)) <= 0.01 and stored_expected == expected_indices and len(stored_saved) == len(stored_expected):
                # manifest matches expected picks and fps; return existing saved files
                existing_files = sorted(output_dir.glob('frame_*.jpg'))
                return existing_files
        except Exception:
            pass

    # If overwrite requested, clear existing matching frame files only (non-destructive)
    if overwrite:
        for path in output_dir.glob('frame_*.jpg'):
            try:
                path.unlink()
            except Exception:
                pass

    probe = subprocess.run(
        [
            'ffprobe', '-v', 'error',
            '-select_streams', 'v:0',
            '-show_entries', 'stream=width,height',
            '-of', 'csv=p=0',
            str(video_path),
        ],
        capture_output=True,
        text=True,
        timeout=20,
    )
    if probe.returncode != 0 or not probe.stdout.strip():
        raise RuntimeError(f'ffprobe failed for {video_path.name}: {probe.stderr.strip()}')
    width, height = map(int, probe.stdout.strip().split(','))
    frame_size = width * height * 3

    cmd = [
        'ffmpeg', '-hide_banner', '-loglevel', 'error',
        '-i', str(video_path),
        '-f', 'rawvideo', '-pix_fmt', 'rgb24',
        'pipe:1',
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if proc.stdout is None:
        raise RuntimeError(f'Could not open ffmpeg pipe for {video_path.name}')

    saved_paths: List[Path] = []
    expected_set = set(int(p) for p in expected_indices)
    expected_last = int(expected_indices[-1]) if len(expected_indices) else -1
    frame_idx = 0

    try:
        while True:
            raw = proc.stdout.read(frame_size)
            if len(raw) < frame_size:
                break
            if frame_idx in expected_set:
                frame_path = output_dir / f'frame_{frame_idx:06d}.jpg'
                Image.frombytes('RGB', (width, height), raw).save(frame_path, quality=95)
                saved_paths.append(frame_path)
                if len(saved_paths) == len(expected_indices):
                    break
            frame_idx += 1
            if frame_idx > expected_last + 10_000:
                raise RuntimeError(f'Exceeded expected frame range for {video_path.name}')
    finally:
        try:
            proc.stdout.close()
        except Exception:
            pass
        proc.wait(timeout=30)

    actual_indices = [int(path.stem.split('_')[1]) for path in saved_paths]
    if actual_indices != expected_indices:
        raise AssertionError(
            f'Frame mismatch for {video_path.name}: expected {len(expected_indices)} frames, got {len(actual_indices)}'
        )

    # write manifest for resume
    manifest = {
        'native_fps': float(native_fps),
        'expected_indices': expected_indices,
        'saved_indices': actual_indices,
        'generated_at': pd.Timestamp.now().isoformat(),
    }
    _save_frame_manifest(output_dir, manifest)

    return saved_paths


def build_frame_record(entry: Dict, saved_paths: List[Path]) -> Dict:
    saved_indices = [int(path.stem.split('_')[1]) for path in saved_paths]
    expected_indices = expected_frame_indices(entry['picks'], entry['native_fps'], entry['duration_limit_sec']).astype(int).tolist()
    exact_match = saved_indices == expected_indices
    return {
        'dataset': entry['dataset'],
        'h5_key': entry['h5_key'],
        'canonical_name': entry['canonical_name'],
        'source_path': str(entry['source_path']),
        'frame_dir': str(saved_paths[0].parent if saved_paths else FRAME_ROOT / entry['dataset'] / entry['canonical_name']),
        'native_fps': entry['native_fps'],
        'actual_duration_sec': entry['actual_duration_sec'],
        'reference_duration_sec': entry['reference_duration_sec'],
        'duration_limit_sec': entry['duration_limit_sec'],
        'expected_frames': len(expected_indices),
        'saved_frames': len(saved_paths),
        'frame_ok': exact_match,
        'expected_indices': expected_indices,
        'saved_indices': saved_indices,
    }


def extract_dataset_frames(entries: List[Dict], overwrite: bool = True) -> pd.DataFrame:
    records = []
    label = entries[0]['dataset'] if entries else 'dataset'
    for entry in tqdm(entries, desc=f'Extracting {label} frames'):
        frame_dir = FRAME_ROOT / entry['dataset'] / entry['canonical_name']
        saved_paths = extract_exact_frames(
            video_path=Path(entry['source_path']),
            output_dir=frame_dir,
            picks=entry['picks'],
            native_fps=entry['native_fps'],
            duration_limit_sec=entry['duration_limit_sec'],
            overwrite=overwrite,
        )
        record = build_frame_record(entry, saved_paths)
        if not record['frame_ok']:
            raise AssertionError(f'Frame verification failed for {entry["canonical_name"]}')
        records.append(record)
    return pd.DataFrame(records)


def show_frame_check(df: pd.DataFrame, dataset_name: str) -> None:
    print()
    print(f'{dataset_name.upper()} frame check')
    print(f'  videos checked : {len(df)}')
    print(f'  frame_ok       : {int(df["frame_ok"].sum())}/{len(df)}')
    print(f'  expected frames: {df["expected_frames"].min()} .. {df["expected_frames"].max()}')
    print(f'  saved frames   : {df["saved_frames"].min()} .. {df["saved_frames"].max()}')


def compare_picks(df: pd.DataFrame, dataset_name: str) -> None:
    print()
    print(f'{dataset_name.upper()} picks preview')
    for _, row in df.head(3).iterrows():
        print(f"  {row['canonical_name']}: {row['saved_frames']} frames, first picks {row['saved_indices'][:10]}")


# Note: caller must explicitly call clean_output_tree(confirm=True) to remove or backup outputs
# clean_output_tree()


## 4. Load models and embed the verified frames

The embedding step uses only the verified frame directories. Each embedding file is written under the canonical dataset name so downstream validation and H5 reconstruction use the same identifiers as the H5 contract.

In [8]:
# ============================================================
# CELL 4 - Model loading and frame embedding
# ============================================================

def build_clip_bundle(model_name: str, pretrained: str) -> Dict:
    model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
    model = model.to(DEVICE).eval()
    expected_dim = 768 if model_name == 'ViT-L-14' else 1024
    return {
        'kind': 'clip',
        'name': model_name,
        'pretrained': pretrained,
        'model': model,
        'preprocess': preprocess,
        'expected_dim': expected_dim,
    }


def build_dinov2_bundle() -> Dict:
    model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14')
    model = model.to(DEVICE).eval()
    preprocess = transforms.Compose([
        transforms.Resize(518, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(518),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return {
        'kind': 'dinov2',
        'name': 'dinov2_vitl14',
        'model': model,
        'preprocess': preprocess,
        'expected_dim': 1024,
    }


def load_image_batch(image_paths: List[Path], preprocess) -> Tuple[Optional[torch.Tensor], List[str]]:
    tensors, names = [], []
    for path in image_paths:
        try:
            tensors.append(preprocess(Image.open(path).convert('RGB')))
            names.append(path.name)
        except Exception as exc:
            print(f'[WARN] Skipping {path.name}: {exc}')
    if not tensors:
        return None, []
    return torch.stack(tensors), names


@torch.no_grad()
def embed_frame_dir(frame_dir: Path, bundle: Dict, batch_size: int = 64) -> Tuple[np.ndarray, List[str]]:
    frame_paths = sorted(frame_dir.glob('frame_*.jpg'))
    if not frame_paths:
        return np.empty((0, bundle['expected_dim']), dtype=np.float32), []

    all_embeddings, all_names = [], []
    for start in range(0, len(frame_paths), batch_size):
        batch_tensor, batch_names = load_image_batch(
            frame_paths[start:start + batch_size],
            bundle['preprocess'],
        )
        if batch_tensor is None:
            continue
        batch_tensor = batch_tensor.to(DEVICE)
        if bundle['kind'] == 'clip':
            embeddings = bundle['model'].encode_image(batch_tensor)
        else:
            embeddings = bundle['model'](batch_tensor)
        all_embeddings.append(embeddings.float().cpu().numpy())
        all_names.extend(batch_names)

    if not all_embeddings:
        return np.empty((0, bundle['expected_dim']), dtype=np.float32), []

    return np.concatenate(all_embeddings, axis=0).astype(np.float32), all_names


def save_embedding_npz(output_path: Path, embeddings: np.ndarray, frame_names: List[str], meta: Dict, model_name: str) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        'embeddings': embeddings.astype(np.float32),
        'frame_names': np.array(frame_names, dtype=object),
        'model_name': np.array(model_name),
        'dataset': np.array(meta['dataset']),
        'h5_key': np.array(meta['h5_key']),
        'canonical_name': np.array(meta['canonical_name']),
        'source_path': np.array(str(meta['source_path'])),
        'native_fps': np.array(meta['native_fps']),
        'actual_duration_sec': np.array(meta['actual_duration_sec']),
        'reference_duration_sec': np.array(meta['reference_duration_sec']),
        'duration_limit_sec': np.array(meta['duration_limit_sec']),
        'n_frames': np.array(meta['n_frames']),
        'n_steps': np.array(meta['n_steps']),
        'picks': meta['picks'].astype(np.int32),
    }
    np.savez_compressed(output_path, **payload)


def read_h5_metadata(h5_path: Path, h5_key: str) -> Dict:
    with h5py.File(h5_path, 'r') as h5:
        group = h5[h5_key]
        meta = {
            'h5_key': h5_key,
            'n_frames': int(group['n_frames'][()]),
            'n_steps': int(group['n_steps'][()]),
            'picks': group['picks'][()].astype(np.int32),
            'gtscore': group['gtscore'][()].astype(np.float32),
            'gtsummary': group['gtsummary'][()],
            'change_points': group['change_points'][()].astype(np.int32),
            'n_frame_per_seg': group['n_frame_per_seg'][()].astype(np.int32),
        }
        if 'user_summary' in group:
            meta['user_summary'] = group['user_summary'][()].astype(np.float32)
        if 'video_name' in group:
            meta['video_name'] = decode_scalar(group['video_name'][()])
    return meta


def _validate_existing_npz(npz_path: Path, expected_count: int, expected_dim: int) -> bool:
    try:
        data = np.load(npz_path, allow_pickle=True)
        if 'embeddings' not in data:
            return False
        emb = data['embeddings']
        if emb.ndim != 2:
            return False
        if emb.shape[0] != expected_count:
            return False
        if emb.shape[1] != expected_dim:
            return False
        return True
    except Exception:
        return False


def embed_dataset(entries: List[Dict], model_bundles: Dict[str, Dict], overwrite: bool = True) -> pd.DataFrame:
    records = []
    dataset_label = entries[0]['dataset'] if entries else 'dataset'
    for entry in tqdm(entries, desc=f'Embedding {dataset_label} videos'):
        frame_dir = FRAME_ROOT / entry['dataset'] / entry['canonical_name']
        if not frame_dir.exists():
            raise FileNotFoundError(f'Missing frame directory: {frame_dir}')

        frame_files = sorted(frame_dir.glob('frame_*.jpg'))
        expected_picks = expected_frame_indices(entry['picks'], entry['native_fps'], entry['duration_limit_sec'])
        expected_count = len(expected_picks)
        actual_indices = [int(path.stem.split('_')[1]) for path in frame_files]
        if actual_indices != expected_picks.astype(int).tolist():
            raise AssertionError(f'Frame verification failed before embedding for {entry["canonical_name"]}')

        meta = read_h5_metadata(TVSUM_H5 if entry['dataset'] == 'tvsum' else SUMME_H5, entry['h5_key'])
        meta.update({
            'dataset': entry['dataset'],
            'canonical_name': entry['canonical_name'],
            'source_path': str(entry['source_path']),
            'native_fps': entry['native_fps'],
            'actual_duration_sec': entry['actual_duration_sec'],
            'reference_duration_sec': entry['reference_duration_sec'],
            'duration_limit_sec': entry['duration_limit_sec'],
        })

        for model_name, bundle in model_bundles.items():
            output_path = EMBEDDINGS_ROOT / entry['dataset'] / model_name / f"{entry['canonical_name']}.npz"
            expected_dim = bundle.get('expected_dim', MODEL_SPECS.get(model_name, {}).get('expected_dim'))

            if output_path.exists() and not overwrite:
                ok = _validate_existing_npz(output_path, expected_count, expected_dim)
                if ok:
                    records.append({
                        'dataset': entry['dataset'],
                        'canonical_name': entry['canonical_name'],
                        'h5_key': entry['h5_key'],
                        'model': model_name,
                        'output_path': str(output_path),
                        'status': 'skipped',
                    })
                    continue
                else:
                    # remove corrupted or mismatched file and re-run
                    try:
                        output_path.unlink()
                    except Exception:
                        pass

            embeddings, frame_names = embed_frame_dir(frame_dir, bundle, batch_size=16)
            if len(frame_names) != expected_count:
                raise AssertionError(f'Embedding count mismatch for {entry["canonical_name"]} / {model_name}')
            save_embedding_npz(output_path, embeddings, frame_names, meta, model_name)
            records.append({
                'dataset': entry['dataset'],
                'canonical_name': entry['canonical_name'],
                'h5_key': entry['h5_key'],
                'model': model_name,
                'output_path': str(output_path),
                'status': 'saved',
                'embedding_shape': embeddings.shape,
                'frame_count': len(frame_names),
                'expected_frame_count': expected_count,
            })
    return pd.DataFrame(records)

clip_vitl14_bundle = build_clip_bundle('ViT-L-14', 'openai')
clip_rn50x64_bundle = build_clip_bundle('RN50x64', 'openai')
dinov2_bundle = build_dinov2_bundle()
MODEL_BUNDLES = {
    'clip_vitl14': clip_vitl14_bundle,
    'clip_rn50x64': clip_rn50x64_bundle,
    'dinov2_vitl14': dinov2_bundle,
}

for name, bundle in MODEL_BUNDLES.items():
    print(f'{name}: {bundle["kind"]} on {DEVICE} (dim={bundle["expected_dim"]})')

summe_embedding_log = embed_dataset(summe_entries, MODEL_BUNDLES, overwrite=False)
tvsum_embedding_log = embed_dataset(tvsum_entries, MODEL_BUNDLES, overwrite=False)

print()
print('Embedding logs')
display(summe_embedding_log.head())
display(tvsum_embedding_log.head())

summe_embedding_log.to_csv(MANIFEST_ROOT / 'summe_embedding_log.csv', index=False)
tvsum_embedding_log.to_csv(MANIFEST_ROOT / 'tvsum_embedding_log.csv', index=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/2.49G [00:00<?, ?B/s]

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


...

Embedding summe videos:   0%|          | 0/25 [00:00<?, ?it/s]

Embedding tvsum videos:   0%|          | 0/50 [00:00<?, ?it/s]


Embedding logs


,dataset,canonical_name,h5_key,model,output_path,status
0,summe,Air_Force_One,video_1,clip_vitl14,/content/drive/MyDrive/4th year/S2/RL/Project/...,skipped
1,summe,Air_Force_One,video_1,clip_rn50x64,/content/drive/MyDrive/4th year/S2/RL/Project/...,skipped
2,summe,Air_Force_One,video_1,dinov2_vitl14,/content/drive/MyDrive/4th year/S2/RL/Project/...,skipped
3,summe,Base jumping,video_2,clip_vitl14,/content/drive/MyDrive/4th year/S2/RL/Project/...,skipped
4,summe,Base jumping,video_2,clip_rn50x64,/content/drive/MyDrive/4th year/S2/RL/Project/...,skipped


,dataset,canonical_name,h5_key,model,output_path,status,embedding_shape,frame_count,expected_frame_count
0,tvsum,AwmHb44_ouw,video_1,clip_vitl14,/content/drive/MyDrive/4th year/S2/RL/Project/...,skipped,NaN,NaN,NaN
1,tvsum,AwmHb44_ouw,video_1,clip_rn50x64,/content/drive/MyDrive/4th year/S2/RL/Project/...,skipped,NaN,NaN,NaN
2,tvsum,AwmHb44_ouw,video_1,dinov2_vitl14,/content/drive/MyDrive/4th year/S2/RL/Project/...,skipped,NaN,NaN,NaN
3,tvsum,98MoyGZKHXc,video_2,clip_vitl14,/content/drive/MyDrive/4th year/S2/RL/Project/...,skipped,NaN,NaN,NaN
4,tvsum,98MoyGZKHXc,video_2,clip_rn50x64,/content/drive/MyDrive/4th year/S2/RL/Project/...,skipped,NaN,NaN,NaN


## 5. Reconstruct the H5 files from the embeddings

For every dataset and model, the original ECCV16 groups are copied and only `features` is replaced by the new embedding tensor. All annotation fields remain aligned to the extracted frame list.

In [14]:
# ============================================================
# CELL 4.5 - Rename existing embedding files to canonical IDs (dry-run first)
# ============================================================

def rename_embeddings_for_dataset(dataset_name: str, dry_run: bool = True, backup: bool = True,
                                  remove_duplicates: bool = False, force_overwrite: bool = False) -> Dict:
    """Rename embeddings to match the H5 canonical ids with safe backups.

    Args:
      dataset_name: 'tvsum' or 'summe'
      dry_run: when True only preview planned actions and write a manifest
      backup: when performing actual changes, copy originals into a timestamped backup
      remove_duplicates: if True, remove source files when destination already exists and is identical
      force_overwrite: if True, overwrite existing destination files with the source (backing up the previous dst first)

    Behavior:
    - Matches candidate files by exact stem match or substring match (handles common naming variants).
    - By default, never deletes or overwrites files; it only reports planned rename actions.
    - When `dry_run=False` it performs renames and (optionally) backups according to flags.

    Returns a dictionary with keys: `mappings`, `skipped`, `conflicts`, `errors`, `manifest`.
    """
    results = {'mappings': [], 'skipped': [], 'conflicts': [], 'errors': [], 'manifest': None}
    timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')

    entries = tvsum_entries if dataset_name == 'tvsum' else summe_entries
    for model_name in MODEL_BUNDLES.keys():
        emb_dir = EMBEDDINGS_ROOT / dataset_name / model_name
        if not emb_dir.exists():
            print(f'[SKIP] missing dir: {emb_dir}')
            results['skipped'].append({'model': model_name, 'reason': 'missing_dir'})
            continue

        print(f'Processing {emb_dir}')
        backup_root = EMBEDDINGS_ROOT.parent / 'rename_backups' / timestamp / dataset_name / model_name
        for entry in entries:
            h5k = entry['h5_key']
            canonical = entry['canonical_name']
            dst = emb_dir / f"{canonical}.npz"

            # Find candidate files that look like the source
            candidates = []
            for p in emb_dir.glob('*.npz'):
                stem = p.stem
                if stem == h5k or stem == canonical or h5k in stem or canonical in stem:
                    candidates.append(p)

            if not candidates:
                # No obvious candidate; record and continue
                results['skipped'].append({'h5_key': h5k, 'canonical': canonical, 'model': model_name, 'reason': 'no_candidate'})
                continue

            # Prefer exact h5_key match, else prefer exact canonical, else first candidate
            src = None
            for p in candidates:
                if p.stem == h5k:
                    src = p
                    break
            if src is None:
                for p in candidates:
                    if p.stem == canonical:
                        src = p
                        break
            if src is None:
                src = candidates[0]

            try:
                if dst.exists():
                    # Destination exists. Do NOT delete source by default.
                    if src.stat().st_size == dst.stat().st_size:
                        print(f'  DUPLICATE: {dst.name} exists and equal size; skipping rename of {src.name}')
                        results['skipped'].append({'src': str(src), 'dst': str(dst), 'model': model_name, 'reason': 'duplicate'})
                        if not dry_run and remove_duplicates:
                            # Optional: remove the redundant source after backing it up
                            if backup:
                                backup_root.mkdir(parents=True, exist_ok=True)
                                try:
                                    shutil.copy2(src, backup_root / src.name)
                                except Exception as exc:
                                    print(f'    WARN: backup failed for {src}: {exc}')
                            try:
                                src.unlink()
                                results['mappings'].append({'action': 'remove_src', 'src': str(src), 'dst': str(dst), 'model': model_name})
                                print(f'    Removed source {src.name} (duplicate)')
                            except Exception as exc:
                                results['errors'].append({'src': str(src), 'error': str(exc)})
                        continue
                    else:
                        # Sizes differ: possible conflict
                        if force_overwrite:
                            print(f'  OVERWRITE: {dst.name} exists and differs; will overwrite with {src.name}')
                            results['mappings'].append({'action': 'overwrite', 'src': str(src), 'dst': str(dst), 'model': model_name})
                            if not dry_run:
                                # backup dst first
                                if backup:
                                    backup_root.mkdir(parents=True, exist_ok=True)
                                    try:
                                        shutil.copy2(dst, backup_root / dst.name)
                                    except Exception as exc:
                                        print(f'    WARN: backup failed for dst {dst}: {exc}')
                                try:
                                    # perform overwrite by removing dst then renaming src
                                    dst.unlink()
                                    src.rename(dst)
                                except Exception as exc:
                                    print(f'    FAILED to overwrite {dst} with {src}: {exc}')
                                    results['errors'].append({'src': str(src), 'dst': str(dst), 'error': str(exc)})
                            continue
                        else:
                            print(f'  CONFLICT: {dst.name} exists and differs; skipping {src.name}')
                            results['conflicts'].append({'src': str(src), 'dst': str(dst), 'model': model_name})
                            continue

                # Plan rename (destination does not exist)
                results['mappings'].append({'action': 'rename', 'src': str(src), 'dst': str(dst), 'model': model_name})
                print(f'  RENAME: {src.name} -> {dst.name}')

                if not dry_run:
                    # ensure parent and backup
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    if backup:
                        backup_root.mkdir(parents=True, exist_ok=True)
                        try:
                            shutil.copy2(src, backup_root / src.name)
                        except Exception as exc:
                            print(f'    WARN: backup failed for {src}: {exc}')
                    try:
                        src.rename(dst)
                    except Exception as exc:
                        print(f'    FAILED to rename {src} -> {dst}: {exc}')
                        results['errors'].append({'src': str(src), 'dst': str(dst), 'error': str(exc)})
            except Exception as exc:
                results['errors'].append({'src': str(src) if 'src' in locals() else None, 'error': str(exc)})

    # Write manifest when performing the actual run or to record dry-run results
    manifest_path = MANIFEST_ROOT / f'rename_manifest_{dataset_name}_{timestamp}.json'
    try:
        manifest_path.parent.mkdir(parents=True, exist_ok=True)
        manifest_payload = {
            'dataset': dataset_name,
            'timestamp': timestamp,
            'dry_run': bool(dry_run),
            'remove_duplicates': bool(remove_duplicates),
            'force_overwrite': bool(force_overwrite),
            'mappings': results['mappings'],
            'skipped': results['skipped'],
            'conflicts': results['conflicts'],
            'errors': results['errors'],
        }
        manifest_path.write_text(json.dumps(manifest_payload, indent=2, ensure_ascii=False), encoding='utf-8')
        results['manifest'] = str(manifest_path)
        print(f'Wrote manifest: {manifest_path}')
    except Exception as exc:
        print(f'Could not write manifest: {exc}')
        results['errors'].append({'manifest_write': str(exc)})

    # Summary
    print('\nSummary:')
    print(f"  planned renames: {len([m for m in results['mappings'] if m.get('action')=='rename'])}")
    print(f"  remove_src actions: {len([m for m in results['mappings'] if m.get('action')=='remove_src'])}")
    print(f"  overwrites: {len([m for m in results['mappings'] if m.get('action')=='overwrite'])}")
    print(f"  conflicts: {len(results['conflicts'])}")
    print(f"  skipped: {len(results['skipped'])}")
    print(f"  errors: {len(results['errors'])}")

    return results


# Usage examples:
# 1) Preview:
rename_embeddings_for_dataset('tvsum', dry_run=True)
# 2) Apply renames without deleting duplicates: rename_embeddings_for_dataset('tvsum', dry_run=False)
# 3) Apply renames and remove duplicate sources (after backing up): rename_embeddings_for_dataset('tvsum', dry_run=False, remove_duplicates=True)
# 4) Force overwrite existing destinations with sources (backups made): rename_embeddings_for_dataset('tvsum', dry_run=False, force_overwrite=True)
# A manifest will be written to MANIFEST_ROOT describing planned/performed actions.


Processing /content/drive/MyDrive/4th year/S2/RL/Project/benchmark_embeddings/embeddings/tvsum/clip_vitl14
  DUPLICATE: AwmHb44_ouw.npz exists and equal size; skipping rename of AwmHb44_ouw.npz
  DUPLICATE: 98MoyGZKHXc.npz exists and equal size; skipping rename of 98MoyGZKHXc.npz
  DUPLICATE: J0nA4VgnoCo.npz exists and equal size; skipping rename of J0nA4VgnoCo.npz
  DUPLICATE: gzDbaEs1Rlg.npz exists and equal size; skipping rename of gzDbaEs1Rlg.npz
  DUPLICATE: XzYM3PfTM4w.npz exists and equal size; skipping rename of XzYM3PfTM4w.npz
  DUPLICATE: HT5vyqe0Xaw.npz exists and equal size; skipping rename of HT5vyqe0Xaw.npz
  DUPLICATE: sTEELN-vY30.npz exists and equal size; skipping rename of sTEELN-vY30.npz
  DUPLICATE: vdmoEJ5YbrQ.npz exists and equal size; skipping rename of vdmoEJ5YbrQ.npz
  DUPLICATE: xwqBXPGE9pQ.npz exists and equal size; skipping rename of xwqBXPGE9pQ.npz
  DUPLICATE: akI8YFjEmUw.npz exists and equal size; skipping rename of akI8YFjEmUw.npz
  DUPLICATE: i3wAGJaakt

{'mappings': [],
 'skipped': [{'src': '/content/drive/MyDrive/4th year/S2/RL/Project/benchmark_embeddings/embeddings/tvsum/clip_vitl14/AwmHb44_ouw.npz',
   'dst': '/content/drive/MyDrive/4th year/S2/RL/Project/benchmark_embeddings/embeddings/tvsum/clip_vitl14/AwmHb44_ouw.npz',
   'model': 'clip_vitl14',
   'reason': 'duplicate'},
  {'src': '/content/drive/MyDrive/4th year/S2/RL/Project/benchmark_embeddings/embeddings/tvsum/clip_vitl14/98MoyGZKHXc.npz',
   'dst': '/content/drive/MyDrive/4th year/S2/RL/Project/benchmark_embeddings/embeddings/tvsum/clip_vitl14/98MoyGZKHXc.npz',
   'model': 'clip_vitl14',
   'reason': 'duplicate'},
  {'src': '/content/drive/MyDrive/4th year/S2/RL/Project/benchmark_embeddings/embeddings/tvsum/clip_vitl14/J0nA4VgnoCo.npz',
   'dst': '/content/drive/MyDrive/4th year/S2/RL/Project/benchmark_embeddings/embeddings/tvsum/clip_vitl14/J0nA4VgnoCo.npz',
   'model': 'clip_vitl14',
   'reason': 'duplicate'},
  {'src': '/content/drive/MyDrive/4th year/S2/RL/Project/ben

In [17]:
# ============================================================
# CELL 5 - Reconstruct H5 files from the new embeddings (updated)
# ============================================================

def create_dataset_safe(group, name, data, compress: bool = True):
    array = np.asarray(data)
    if array.shape == ():
        group.create_dataset(name, data=array)
    else:
        kwargs = {'data': array}
        if compress:
            kwargs['compression'] = 'gzip'
        group.create_dataset(name, **kwargs)


def build_h5_for_model(dataset_name: str, model_name: str, original_h5: Path, out_h5: Path) -> None:
    model_embeddings_dir = EMBEDDINGS_ROOT / dataset_name / model_name
    out_h5.parent.mkdir(parents=True, exist_ok=True)

    annotation_fields = [
        'picks',
        'gtscore',
        'gtsummary',
        'user_summary',
        'change_points',
        'n_frame_per_seg',
        'n_frames',
        # 'n_steps',  <-- Removed from here
    ]

    with h5py.File(original_h5, 'r') as src_h5, h5py.File(out_h5, 'w') as dst_h5:
        for h5_key in h5_sorted_keys(src_h5):
            src_grp = src_h5[h5_key]
            dst_grp = dst_h5.create_group(h5_key)

            for field_name in annotation_fields:
                if field_name in src_grp:
                    create_dataset_safe(dst_grp, field_name, src_grp[field_name][()], compress=True)

            # H5-provided video name (may be 'video_1' style) if present
            h5_video_name = decode_scalar(src_grp['video_name'][()]) if 'video_name' in src_grp else None

            # Determine canonical embedding name using available mappings (prefer embedding filenames)
            canonical_from_entries = None
            if dataset_name == 'tvsum' and 'tvsum_entries' in globals():
                try:
                    canonical_from_entries = next(e['canonical_name'] for e in tvsum_entries if e['h5_key'] == h5_key)
                except StopIteration:
                    canonical_from_entries = None
            if dataset_name == 'summe' and 'summe_entries' in globals():
                try:
                    canonical_from_entries = next(e['canonical_name'] for e in summe_entries if e['h5_key'] == h5_key)
                except StopIteration:
                    canonical_from_entries = canonical_from_entries

            # Candidate embedding basenames in preference order: mapping -> h5 video_name -> h5_key
            candidate_names = []
            if canonical_from_entries:
                candidate_names.append(canonical_from_entries)
            if h5_video_name and h5_video_name not in candidate_names:
                candidate_names.append(h5_video_name)
            if h5_key not in candidate_names:
                candidate_names.append(h5_key)

            emb_path = None
            tried = []
            used_canonical = None
            for cname in candidate_names:
                p = model_embeddings_dir / f"{cname}.npz"
                tried.append(str(p))
                if p.exists():
                    emb_path = p
                    used_canonical = cname
                    break

            if emb_path is None:
                raise FileNotFoundError(
                    f"Missing embedding for H5 key '{h5_key}'. Tried: {tried}")

            # Preserve original H5 video_name if present
            if h5_video_name is not None:
                try:
                    dst_grp.create_dataset('video_name', data=np.bytes_(h5_video_name))
                except Exception:
                    pass

            # Store both the H5 group key (video number) and the chosen embedding id
            try:
                dst_grp.create_dataset('video_number', data=np.bytes_(h5_key))
                dst_grp.create_dataset('video_id', data=np.bytes_(used_canonical))
            except Exception:
                pass

            arr = np.load(emb_path, allow_pickle=True)
            feats = arr['embeddings'] if 'embeddings' in arr else None
            if feats is None:
                raise ValueError(f'No embeddings array in {emb_path}')
            create_dataset_safe(dst_grp, 'features', feats.astype(np.float32), compress=True)

            # Explicitly set n_steps to the number of actual embeddings
            create_dataset_safe(dst_grp, 'n_steps', np.array(feats.shape[0]), compress=True)


    print(f'Wrote {out_h5}')


def validate_reconstructed_h5(dataset_name: str, model_name: str, original_h5: Path, reconstructed_h5: Path) -> None:
    with h5py.File(original_h5, 'r') as src_h5, h5py.File(reconstructed_h5, 'r') as dst_h5:
        src_keys = h5_sorted_keys(src_h5)
        dst_keys = h5_sorted_keys(dst_h5)
        if src_keys != dst_keys:
            raise AssertionError(f'H5 key mismatch for {dataset_name}/{model_name}')
        for h5_key in src_keys:
            src_grp = src_h5[h5_key]
            dst_grp = dst_h5[h5_key]

            # Prefer the chosen embedding id recorded in the reconstructed H5 ('video_id'),
            # then fall back to the original H5 'video_name', then to h5_key.
            emb_basename = None
            if 'video_id' in dst_grp:
                emb_basename = decode_scalar(dst_grp['video_id'][()])
            elif 'video_name' in src_grp:
                emb_basename = decode_scalar(src_grp['video_name'][()])
            else:
                emb_basename = h5_key

            emb_path = EMBEDDINGS_ROOT / dataset_name / model_name / f"{emb_basename}.npz"

            # If emb_path is missing, try to resolve via the entries mapping (safe fallback)
            if not emb_path.exists():
                alt = None
                if dataset_name == 'tvsum' and 'tvsum_entries' in globals():
                    try:
                        alt = next(e['canonical_name'] for e in tvsum_entries if e['h5_key'] == h5_key)
                    except StopIteration:
                        alt = None
                if dataset_name == 'summe' and 'summe_entries' in globals() and alt is None:
                    try:
                        alt = next(e['canonical_name'] for e in summe_entries if e['h5_key'] == h5_key)
                    except StopIteration:
                        alt = None
                if alt:
                    alt_path = EMBEDDINGS_ROOT / dataset_name / model_name / f"{alt}.npz"
                    if alt_path.exists():
                        emb_path = alt_path
                    else:
                        raise FileNotFoundError(f"Missing embedding for {h5_key}; tried {emb_path} and {alt_path}")
                else:
                    raise FileNotFoundError(f"Missing embedding for {h5_key}; tried {emb_path}")

            emb = np.load(emb_path, allow_pickle=True)['embeddings']
            if tuple(dst_grp['features'].shape) != tuple(emb.shape):
                raise AssertionError(f'Feature shape mismatch in {dataset_name}/{model_name}/{h5_key}')
            if int(dst_grp['n_steps'][()]) != emb.shape[0]:
                raise AssertionError(f'n_steps mismatch in {dataset_name}/{model_name}/{h5_key}')


def reconstruct_dataset_h5(dataset_name: str, original_h5: Path) -> pd.DataFrame:
    records = []
    for model_name in MODEL_BUNDLES.keys():
        out_h5 = H5_OUT_ROOT / f'eccv16_dataset_{dataset_name}_{model_name}.h5'
        build_h5_for_model(dataset_name, model_name, original_h5, out_h5)
        validate_reconstructed_h5(dataset_name, model_name, original_h5, out_h5)
        records.append({
            'dataset': dataset_name,
            'model': model_name,
            'output_h5': str(out_h5),
            'status': 'saved',
        })
    return pd.DataFrame(records)


summe_h5_manifest = reconstruct_dataset_h5('summe', SUMME_H5)
# tvsum will be created after ensuring embedding filenames match the canonical ids
tvsum_h5_manifest = reconstruct_dataset_h5('tvsum', TVSUM_H5)

display(summe_h5_manifest)
summe_h5_manifest.to_csv(MANIFEST_ROOT / 'summe_h5_manifest.csv', index=False)
# tvsum_h5_manifest.to_csv(MANIFEST_ROOT / 'tvsum_h5_manifest.csv', index=False)


Wrote /content/drive/MyDrive/4th year/S2/RL/Project/h5_files/eccv16_dataset_tvsum_clip_vitl14.h5
Wrote /content/drive/MyDrive/4th year/S2/RL/Project/h5_files/eccv16_dataset_tvsum_clip_rn50x64.h5
Wrote /content/drive/MyDrive/4th year/S2/RL/Project/h5_files/eccv16_dataset_tvsum_dinov2_vitl14.h5


,dataset,model,output_h5,status
0,summe,clip_vitl14,/content/drive/MyDrive/4th year/S2/RL/Project/...,saved
1,summe,clip_rn50x64,/content/drive/MyDrive/4th year/S2/RL/Project/...,saved
2,summe,dinov2_vitl14,/content/drive/MyDrive/4th year/S2/RL/Project/...,saved


In [18]:
tvsum_h5_manifest.to_csv(MANIFEST_ROOT / 'tvsum_h5_manifest.csv', index=False)

In [19]:

display(tvsum_h5_manifest)

,dataset,model,output_h5,status
0,tvsum,clip_vitl14,/content/drive/MyDrive/4th year/S2/RL/Project/...,saved
1,tvsum,clip_rn50x64,/content/drive/MyDrive/4th year/S2/RL/Project/...,saved
2,tvsum,dinov2_vitl14,/content/drive/MyDrive/4th year/S2/RL/Project/...,saved


## 6. Final validation and summary

This final check confirms that every expected embedding exists, every reconstructed H5 file is present, and the frame selection remained aligned to `picks` before embedding.

In [21]:
# ============================================================
# CELL 6 - Final validation summary
# ============================================================

def validate_embeddings_on_disk(entries: List[Dict], dataset_name: str) -> pd.DataFrame:
    rows = []
    for model_name in MODEL_BUNDLES.keys():
        model_dir = EMBEDDINGS_ROOT / dataset_name / model_name
        missing = []
        bad_shapes = []
        for entry in entries:
            npz_path = model_dir / f"{entry['canonical_name']}.npz"
            if not npz_path.exists():
                missing.append(entry['canonical_name'])
                continue
            data = np.load(npz_path, allow_pickle=True)
            emb = data['embeddings']
            expected_count = len(expected_frame_indices(entry['picks'], entry['native_fps'], entry['duration_limit_sec']))
            if emb.ndim != 2 or emb.shape[0] != expected_count:
                bad_shapes.append((entry['canonical_name'], tuple(emb.shape)))
            if emb.shape[1] != MODEL_SPECS[model_name]['expected_dim']:
                bad_shapes.append((entry['canonical_name'], tuple(emb.shape)))

        rows.append({
            'dataset': dataset_name,
            'model': model_name,
            'expected_files': len(entries),
            'missing_count': len(missing),
            'bad_count': len(bad_shapes),
            'all_ok': len(missing) == 0 and len(bad_shapes) == 0,
        })
        print(f'{dataset_name}/{model_name}: missing={len(missing)}, bad={len(bad_shapes)}')
        if missing:
            print(f'  sample missing: {missing[:5]}')
        if bad_shapes:
            print(f'  sample bad: {bad_shapes[:5]}')
    return pd.DataFrame(rows)


def validate_h5_outputs() -> pd.DataFrame:
    rows = []
    for dataset_name in ['summe', 'tvsum']:
        for model_name in MODEL_BUNDLES.keys():
            out_h5 = H5_OUT_ROOT / f'eccv16_dataset_{dataset_name}_{model_name}.h5'
            rows.append({
                'dataset': dataset_name,
                'model': model_name,
                'exists': out_h5.exists(),
                'path': str(out_h5),
            })
    return pd.DataFrame(rows)


embedding_check_tvsum = validate_embeddings_on_disk(tvsum_entries, 'tvsum')
embedding_check_summe = validate_embeddings_on_disk(summe_entries, 'summe')
h5_check = validate_h5_outputs()

print()
print('Embedding validation summary')
display(embedding_check_tvsum)
display(embedding_check_summe)
print()
print('H5 output check')
display(h5_check)

print()
print('Summary')
# print('TVSum frame manifests all OK:', bool(tvsum_frame_manifest['frame_ok'].all()))
# print('SumMe frame manifests all OK:', bool(summe_frame_manifest['frame_ok'].all()))
print('TVSum embeddings all OK:', bool(embedding_check_tvsum['all_ok'].all()))
print('SumMe embeddings all OK:', bool(embedding_check_summe['all_ok'].all()))
print('All H5 files exist:', bool(h5_check['exists'].all()))
print()
print('Outputs written to:')
print(f'  Frames: {FRAME_ROOT}')
print(f'  Embeddings: {EMBEDDINGS_ROOT}')
print(f'  H5 files: {H5_OUT_ROOT}')
print(f'  Manifests: {MANIFEST_ROOT}')

tvsum/clip_vitl14: missing=0, bad=0
tvsum/clip_rn50x64: missing=0, bad=0
tvsum/dinov2_vitl14: missing=0, bad=0
summe/clip_vitl14: missing=0, bad=0
summe/clip_rn50x64: missing=0, bad=0
summe/dinov2_vitl14: missing=0, bad=0

Embedding validation summary


,dataset,model,expected_files,missing_count,bad_count,all_ok
0,tvsum,clip_vitl14,50,0,0,True
1,tvsum,clip_rn50x64,50,0,0,True
2,tvsum,dinov2_vitl14,50,0,0,True


,dataset,model,expected_files,missing_count,bad_count,all_ok
0,summe,clip_vitl14,25,0,0,True
1,summe,clip_rn50x64,25,0,0,True
2,summe,dinov2_vitl14,25,0,0,True



H5 output check


,dataset,model,exists,path
0,summe,clip_vitl14,True,/content/drive/MyDrive/4th year/S2/RL/Project/...
1,summe,clip_rn50x64,True,/content/drive/MyDrive/4th year/S2/RL/Project/...
2,summe,dinov2_vitl14,True,/content/drive/MyDrive/4th year/S2/RL/Project/...
3,tvsum,clip_vitl14,True,/content/drive/MyDrive/4th year/S2/RL/Project/...
4,tvsum,clip_rn50x64,True,/content/drive/MyDrive/4th year/S2/RL/Project/...
5,tvsum,dinov2_vitl14,True,/content/drive/MyDrive/4th year/S2/RL/Project/...



Summary
TVSum embeddings all OK: True
SumMe embeddings all OK: True
All H5 files exist: True

Outputs written to:
  Frames: /content/drive/MyDrive/4th year/S2/RL/Project/benchmark_embeddings/frames_2fps
  Embeddings: /content/drive/MyDrive/4th year/S2/RL/Project/benchmark_embeddings/embeddings
  H5 files: /content/drive/MyDrive/4th year/S2/RL/Project/h5_files
  Manifests: /content/drive/MyDrive/4th year/S2/RL/Project/benchmark_embeddings/manifests


## 7. Run the full rebuild

Run this cell after you are ready. It clears the output tree, rebuilds the frame manifests from H5, verifies the extracted frame lists, embeds the selected frames, reconstructs the H5 files, and performs the final validation.

In [ ]:
# Uncomment to run the full rebuild from scratch.
# clean_output_tree()
# tvsum_entries = build_tvsum_entries()
# summe_entries = build_summe_entries()
# tvsum_contract_df = audit_contract(tvsum_entries, 'tvsum')
# summe_contract_df = audit_contract(summe_entries, 'summe')
# tvsum_frame_manifest = extract_dataset_frames(tvsum_entries, overwrite=True)
# summe_frame_manifest = extract_dataset_frames(summe_entries, overwrite=True)
# summe_embedding_log = embed_dataset(summe_entries, MODEL_BUNDLES, overwrite=True)
# tvsum_embedding_log = embed_dataset(tvsum_entries, MODEL_BUNDLES, overwrite=True)
# summe_h5_manifest = reconstruct_dataset_h5('summe', SUMME_H5)
# tvsum_h5_manifest = reconstruct_dataset_h5('tvsum', TVSUM_H5)
# embedding_check_tvsum = validate_embeddings_on_disk(tvsum_entries, 'tvsum')
# embedding_check_summe = validate_embeddings_on_disk(summe_entries, 'summe')

In [24]:
# ============================================================
# CELL 5.1 - Inspect and compare original vs reconstructed H5 files
# ============================================================

def _dataset_value_summary(value: Any) -> str:
    try:
        array = np.asarray(value)
        if array.shape == ():
            return repr(decode_scalar(array))
        return f'shape={array.shape}, dtype={array.dtype}'
    except Exception:
        return f'type={type(value).__name__}'


def _group_summary(group: h5py.Group) -> Dict[str, str]:
    summary = {}
    for key in group.keys():
        try:
            summary[key] = _dataset_value_summary(group[key][()])
        except Exception as exc:
            summary[key] = f'<error reading: {exc}>'
    return summary


def inspect_h5_file(h5_path: Path, label: str, max_groups: int = 5) -> pd.DataFrame:
    rows = []
    with h5py.File(h5_path, 'r') as h5:
        keys = h5_sorted_keys(h5)
        print(f'[{label}] {h5_path}')
        print(f'  groups: {len(keys)}')
        for h5_key in keys[:max_groups]:
            grp = h5[h5_key]
            rows.append({
                'label': label,
                'h5_key': h5_key,
                'video_name': decode_scalar(grp['video_name'][()]) if 'video_name' in grp else None,
                'video_id': decode_scalar(grp['video_id'][()]) if 'video_id' in grp else None,
                'video_number': decode_scalar(grp['video_number'][()]) if 'video_number' in grp else None,
                'n_frames': int(grp['n_frames'][()]) if 'n_frames' in grp else None,
                'n_steps': int(grp['n_steps'][()]) if 'n_steps' in grp else None,
                'fields': ', '.join(list(grp.keys())),
            })
    df = pd.DataFrame(rows)
    if len(df):
        display(df)
    return df


def compare_h5_pair(original_h5: Path, reconstructed_h5: Path, max_diffs: int = 10) -> pd.DataFrame:
    rows = []
    with h5py.File(original_h5, 'r') as src_h5, h5py.File(reconstructed_h5, 'r') as dst_h5:
        src_keys = h5_sorted_keys(src_h5)
        dst_keys = h5_sorted_keys(dst_h5)
        print(f'Original groups      : {len(src_keys)}')
        print(f'Reconstructed groups : {len(dst_keys)}')
        print(f'Keys identical       : {src_keys == dst_keys}')

        for h5_key in src_keys:
            if h5_key not in dst_h5:
                rows.append({'h5_key': h5_key, 'status': 'missing_in_reconstructed'})
                continue

            src_grp = src_h5[h5_key]
            dst_grp = dst_h5[h5_key]
            src_fields = set(src_grp.keys())
            dst_fields = set(dst_grp.keys())

            common_fields = sorted(src_fields & dst_fields)
            missing_in_dst = sorted(src_fields - dst_fields)
            extra_in_dst = sorted(dst_fields - src_fields)

            feature_shape_src = tuple(src_grp['features'].shape) if 'features' in src_grp else None
            feature_shape_dst = tuple(dst_grp['features'].shape) if 'features' in dst_grp else None
            n_steps_src = int(src_grp['n_steps'][()]) if 'n_steps' in src_grp else None
            n_steps_dst = int(dst_grp['n_steps'][()]) if 'n_steps' in dst_grp else None
            video_name_src = decode_scalar(src_grp['video_name'][()]) if 'video_name' in src_grp else None
            video_name_dst = decode_scalar(dst_grp['video_name'][()]) if 'video_name' in dst_grp else None
            video_id_dst = decode_scalar(dst_grp['video_id'][()]) if 'video_id' in dst_grp else None
            video_number_dst = decode_scalar(dst_grp['video_number'][()]) if 'video_number' in dst_grp else None

            status = 'ok'
            if feature_shape_src != feature_shape_dst or n_steps_src != n_steps_dst or missing_in_dst or extra_in_dst:
                status = 'different'

            rows.append({
                'h5_key': h5_key,
                'status': status,
                'video_name_src': video_name_src,
                'video_name_dst': video_name_dst,
                'video_id_dst': video_id_dst,
                'video_number_dst': video_number_dst,
                'feature_shape_src': feature_shape_src,
                'feature_shape_dst': feature_shape_dst,
                'n_steps_src': n_steps_src,
                'n_steps_dst': n_steps_dst,
                'missing_in_dst': ', '.join(missing_in_dst),
                'extra_in_dst': ', '.join(extra_in_dst),
                'shared_fields': ', '.join(common_fields),
            })

    df = pd.DataFrame(rows)
    print(f'Comparison summary: {len(df)} groups checked, {(df["status"] == "different").sum()} differences')
    if len(df):
        display(df.head(max_diffs))
    return df


def inspect_single_group(original_h5: Path, reconstructed_h5: Path, h5_key: str) -> None:
    with h5py.File(original_h5, 'r') as src_h5, h5py.File(reconstructed_h5, 'r') as dst_h5:
        print(f'--- Original: {h5_key}')
        src_grp = src_h5[h5_key]
        print('video_name   :', decode_scalar(src_grp['video_name'][()]) if 'video_name' in src_grp else None)
        print('n_frames     :', int(src_grp['n_frames'][()]) if 'n_frames' in src_grp else None)
        print('n_steps      :', int(src_grp['n_steps'][()]) if 'n_steps' in src_grp else None)
        print('fields       :', list(src_grp.keys()))
        print('features     :', tuple(src_grp['features'].shape) if 'features' in src_grp else None)

        print(f'--- Reconstructed: {h5_key}')
        dst_grp = dst_h5[h5_key]
        print('video_name   :', decode_scalar(dst_grp['video_name'][()]) if 'video_name' in dst_grp else None)
        print('video_id     :', decode_scalar(dst_grp['video_id'][()]) if 'video_id' in dst_grp else None)
        print('video_number :', decode_scalar(dst_grp['video_number'][()]) if 'video_number' in dst_grp else None)
        print('n_frames     :', int(dst_grp['n_frames'][()]) if 'n_frames' in dst_grp else None)
        print('n_steps      :', int(dst_grp['n_steps'][()]) if 'n_steps' in dst_grp else None)
        print('fields       :', list(dst_grp.keys()))
        print('features     :', tuple(dst_grp['features'].shape) if 'features' in dst_grp else None)


# Set these to whichever pair you want to inspect.
# Example for TVSum:
original_h5_to_check = TVSUM_H5
reconstructed_h5_to_check = H5_OUT_ROOT / 'eccv16_dataset_tvsum_clip_vitl14.h5'
h5_key_to_check = 'video_2'

# Example for SumMe:
# original_h5_to_check = SUMME_H5
# reconstructed_h5_to_check = H5_OUT_ROOT / 'eccv16_dataset_summe_clip_vitl14.h5'
# h5_key_to_check = 'video_1'

# Run these after setting the variables above.
original_summary_df = inspect_h5_file(original_h5_to_check, 'original')
reconstructed_summary_df = inspect_h5_file(reconstructed_h5_to_check, 'reconstructed')
comparison_df = compare_h5_pair(original_h5_to_check, reconstructed_h5_to_check)
inspect_single_group(original_h5_to_check, reconstructed_h5_to_check, h5_key_to_check)


[original] /content/drive/MyDrive/4th year/S2/RL/Project/Dataset/TVSUM/eccv16_dataset_tvsum_google_pool5.h5
  groups: 50


,label,h5_key,video_name,video_id,video_number,n_frames,n_steps,fields
0,original,video_1,None,None,None,10597,707,"change_points, features, gtscore, gtsummary, n..."
1,original,video_2,None,None,None,4688,313,"change_points, features, gtscore, gtsummary, n..."
2,original,video_3,None,None,None,14019,935,"change_points, features, gtscore, gtsummary, n..."
3,original,video_4,None,None,None,7210,481,"change_points, features, gtscore, gtsummary, n..."
4,original,video_5,None,None,None,3327,222,"change_points, features, gtscore, gtsummary, n..."


[reconstructed] /content/drive/MyDrive/4th year/S2/RL/Project/h5_files/eccv16_dataset_tvsum_clip_vitl14.h5
  groups: 50


,label,h5_key,video_name,video_id,video_number,n_frames,n_steps,fields
0,reconstructed,video_1,None,AwmHb44_ouw,video_1,10597,707,"change_points, features, gtscore, gtsummary, n..."
1,reconstructed,video_2,None,98MoyGZKHXc,video_2,4688,312,"change_points, features, gtscore, gtsummary, n..."
2,reconstructed,video_3,None,J0nA4VgnoCo,video_3,14019,934,"change_points, features, gtscore, gtsummary, n..."
3,reconstructed,video_4,None,gzDbaEs1Rlg,video_4,7210,481,"change_points, features, gtscore, gtsummary, n..."
4,reconstructed,video_5,None,XzYM3PfTM4w,video_5,3327,222,"change_points, features, gtscore, gtsummary, n..."


Original groups      : 50
Reconstructed groups : 50
Keys identical       : True
Comparison summary: 50 groups checked, 50 differences


,h5_key,status,video_name_src,video_name_dst,video_id_dst,video_number_dst,feature_shape_src,feature_shape_dst,n_steps_src,n_steps_dst,missing_in_dst,extra_in_dst,shared_fields
0,video_1,different,None,None,AwmHb44_ouw,video_1,"(707, 1024)","(707, 768)",707,707,,"video_id, video_number","change_points, features, gtscore, gtsummary, n..."
1,video_2,different,None,None,98MoyGZKHXc,video_2,"(313, 1024)","(312, 768)",313,312,,"video_id, video_number","change_points, features, gtscore, gtsummary, n..."
2,video_3,different,None,None,J0nA4VgnoCo,video_3,"(935, 1024)","(934, 768)",935,934,,"video_id, video_number","change_points, features, gtscore, gtsummary, n..."
3,video_4,different,None,None,gzDbaEs1Rlg,video_4,"(481, 1024)","(481, 768)",481,481,,"video_id, video_number","change_points, features, gtscore, gtsummary, n..."
4,video_5,different,None,None,XzYM3PfTM4w,video_5,"(222, 1024)","(222, 768)",222,222,,"video_id, video_number","change_points, features, gtscore, gtsummary, n..."
5,video_6,different,None,None,HT5vyqe0Xaw,video_6,"(645, 1024)","(644, 768)",645,644,,"video_id, video_number","change_points, features, gtscore, gtsummary, n..."
6,video_7,different,None,None,sTEELN-vY30,video_7,"(298, 1024)","(298, 768)",298,298,,"video_id, video_number","change_points, features, gtscore, gtsummary, n..."
7,video_8,different,None,None,vdmoEJ5YbrQ,video_8,"(658, 1024)","(658, 768)",658,658,,"video_id, video_number","change_points, features, gtscore, gtsummary, n..."
8,video_9,different,None,None,xwqBXPGE9pQ,video_9,"(468, 1024)","(466, 768)",468,466,,"video_id, video_number","change_points, features, gtscore, gtsummary, n..."
9,video_10,different,None,None,akI8YFjEmUw,video_10,"(267, 1024)","(266, 768)",267,266,,"video_id, video_number","change_points, features, gtscore, gtsummary, n..."


--- Original: video_2
video_name   : None
n_frames     : 4688
n_steps      : 313
fields       : ['change_points', 'features', 'gtscore', 'gtsummary', 'n_frame_per_seg', 'n_frames', 'n_steps', 'picks', 'user_summary']
features     : (313, 1024)
--- Reconstructed: video_2
video_name   : None
video_id     : 98MoyGZKHXc
video_number : video_2
n_frames     : 4688
n_steps      : 312
fields       : ['change_points', 'features', 'gtscore', 'gtsummary', 'n_frame_per_seg', 'n_frames', 'n_steps', 'picks', 'user_summary', 'video_id', 'video_number']
features     : (312, 768)
